In [ ]:
import torch
import plotly.graph_objects as go

from src.gp_ccm import run_sigGPCCM_experiment
from src.sp_ccm import run_ccm_experiment
from src.visualise import vis_synchrony

torch.set_printoptions(sci_mode = False)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

In [11]:
# Python package versions used
%load_ext watermark
%watermark --python
%watermark --iversions

Python implementation: CPython
Python version       : 3.11.3
IPython version      : 8.18.0

torch : 2.1.1+cu118
plotly: 5.9.0



# Generate data

In [ ]:
co2_norm = torch.load("data/CO2_vostok_stan_400kyr_timeseries.pt").to(torch.float32)

# Initalise first 3 values
g = torch.tensor([0.2, 0.1, 0.1])

true_offset = -2

for t in range(2, co2_norm.shape[0] + 1):
    
    # Co2 is external forcing 
    g_next = g_next = (g[t] * (1.4 - (1.9 * g[t]) - (0.3 * co2_norm[t + true_offset])))
    g = torch.concat((g, g_next.unsqueeze(0)))

g_norm = g.sub(g.mean(dim = -1).unsqueeze(-1)).div(g.std(dim = -1).unsqueeze(-1))[0:401]

In [ ]:
torch.corrcoef(torch.stack((co2_norm, g_norm)))

In [ ]:
MAX = 401

fig = go.Figure()

fig.add_trace(go.Scatter(x = torch.arange(0, co2_norm.shape[0])[0:MAX], y = co2_norm[0:MAX],
                    mode = 'lines',
                    name = 'F',
                    line_color = "red"))

fig.add_trace(go.Scatter(x = torch.arange(0, co2_norm.shape[0])[0:MAX], y = g_norm[0:MAX],
                    mode = 'lines',
                    name = 'G',
                    line_color = "blue"))

fig.update_layout(title = 'Synchronous time series',
                   xaxis_title = 't',
                   yaxis_title = 'values')

fig.update_layout(template = "plotly_white")
fig.update_layout(font_family = "Lato")
fig.update_layout(xaxis_range=[-2, MAX])

fig.update_layout(autosize = False, width = 1000, height = 400)

fig.show()

In [ ]:
# --- GLOBALS ----#

k = 3
N_TRAIN = torch.tensor([100]).to(device)

##############
### GP-CCM ###
##############

sig_filter = torch.ones(size = (k, )).to(device)
sig_shift = torch.tensor(sig_filter.shape[0] - 1).to(device) + 1 # k -1 

# both methods rely on noise for numerical stability
NOISE_SCALE = torch.tensor([0.05], device = device) # for diagonal
NOISE_SCALE_low = torch.tensor([0.01], device = device) # for diagonal
NOISE_SCALE_medium = torch.tensor([0.02], device = device) # for diagonal

RBF_SCALE = torch.tensor([0.4], device = device)

############
### ECCM ###
############

ccm_filter = torch.ones(size = (k, )).to(device) # same as sig filter
ccm_shift = torch.tensor(ccm_filter.shape[0] - 1).to(device) + 1

# CO2 -> G

In [ ]:
shifts = torch.arange(-8, 8 + 1, 1)

shift_results_gpccm_CO2G = torch.zeros(size = (shifts.shape[0], 3))

for i, s in enumerate(shifts):
    print("Shift", s.item())
    CO2G_gpccm_rho_mean, CO2G_gpccm_rho_sd, CO2G_gpccm_rho_ind_p95 =  run_sigGPCCM_experiment(
        causal_x = co2_norm.to(device),
        causal_y = g_norm.to(device),
        sig_filter = sig_filter.to(device),
        sig_shift = s.to(device), # try a different shift
        rbf_scale = RBF_SCALE, 
        noise_scale = NOISE_SCALE, 
        n_train = N_TRAIN.to(device),
        device = device)

    shift_results_gpccm_CO2G[i, 0] = CO2G_gpccm_rho_mean
    shift_results_gpccm_CO2G[i, 1] = CO2G_gpccm_rho_sd
    shift_results_gpccm_CO2G[i, 2] = CO2G_gpccm_rho_ind_p95

    print(" ")

vis_synchrony(shift_results_gpccm_CO2G, k, shifts, true_offset, True)

In [ ]:
shifts = torch.arange(-8, 8 + 1, 1)

shift_results_ccm_CO2G = torch.zeros(size = (shifts.shape[0], 3))

for i, s in enumerate(shifts):
    print("Shift", s.item())

    CO2G_ccm_rho_mean, CO2G_ccm_rho_sd, CO2G_ccm_rho_ind_p95, CO2G_ccm_noise =  run_ccm_experiment(
        causal_x = co2_norm.to(device),
        causal_y = g_norm.to(device),
        ccm_filter = ccm_filter.to(device),
        ccm_shift = s.to(device), # try a different shift
        n_train = N_TRAIN.to(device),
        device = device)

    shift_results_ccm_CO2G[i, 0] = CO2G_ccm_rho_mean
    shift_results_ccm_CO2G[i, 1] = CO2G_ccm_rho_sd
    shift_results_ccm_CO2G[i, 2] = CO2G_ccm_rho_ind_p95

    print(" ")

# Visualise ccm 
vis_synchrony(shift_results_ccm_CO2G, k, shifts, true_offset, False, legend_y_displacement = 0.01)

# G -> CO2

In [ ]:
# GPCCM loop
shifts = torch.arange(-8, 8 + 1, 1)

shift_results_gpccm_GCO2 = torch.zeros(size = (shifts.shape[0], 3))

for i, s in enumerate(shifts):
    print("Shift", s.item())
    GCO2_gpccm_rho_mean, GCO2_gpccm_rho_sd, GCO2_gpccm_rho_ind_p95 =  run_sigGPCCM_experiment(
        causal_x = g_norm.to(device),
        causal_y = co2_norm.to(device),
        sig_filter = sig_filter.to(device),
        sig_shift = s.to(device), # try a different shift
        rbf_scale = RBF_SCALE, 
        noise_scale = NOISE_SCALE_low, # stable 
        n_train = N_TRAIN.to(device),
        device = device)

    shift_results_gpccm_GCO2[i, 0] = GCO2_gpccm_rho_mean
    shift_results_gpccm_GCO2[i, 1] = GCO2_gpccm_rho_sd
    shift_results_gpccm_GCO2[i, 2] = GCO2_gpccm_rho_ind_p95

    print(" ")

vis_synchrony(shift_results_gpccm_GCO2, k, shifts, true_offset, True)

In [ ]:
# CCM loop
shifts = torch.arange(-8, 8 + 1, 1)

shift_results_ccm_GCO2 = torch.zeros(size = (shifts.shape[0], 3))

for i, s in enumerate(shifts):
    print("Shift", s.item())

    GCO2_ccm_rho_mean, GCO2_ccm_rho_sd, GCO2_ccm_rho_ind_p95, GCO2_ccm_noise =  run_ccm_experiment(
        causal_x = g_norm.to(device),
        causal_y = co2_norm.to(device),
        ccm_filter = ccm_filter.to(device),
        ccm_shift = s.to(device), # try a different shift
        n_train = N_TRAIN.to(device),
        device = device)

    shift_results_ccm_GCO2[i, 0] = GCO2_ccm_rho_mean
    shift_results_ccm_GCO2[i, 1] = GCO2_ccm_rho_sd
    shift_results_ccm_GCO2[i, 2] = GCO2_ccm_rho_ind_p95

    print(" ")

vis_synchrony(shift_results_ccm_GCO2, k, shifts, true_offset, False)

# Save

In [ ]:
# CCM
torch.save(shift_results_ccm_CO2G, "results/synchrony/shift_results_ccm_CO2G.pt")
torch.save(shift_results_ccm_GCO2, "results/synchrony/shift_results_ccm_GCO2.pt")

# GP-CCM
torch.save(shift_results_gpccm_CO2G, "results/synchrony/shift_results_gpccm_CO2G.pt")
torch.save(shift_results_gpccm_GCO2, "results/synchrony/shift_results_gpccm_GCO2.pt")